In [3]:
!pip install -q gradio transformers torch reportlab requests

In [4]:
import gradio as gr
import sqlite3
import re
import torch
import datetime
import urllib.parse
from transformers import AutoTokenizer, AutoModelForCausalLM
from reportlab.pdfgen import canvas

In [5]:
model_name = "ibm-granite/granite-3.3-2b-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model Loaded Successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model Loaded Successfully


In [6]:
SCHEMA = {
    "users": ["id", "name", "email", "signup_date", "age", "country", "status"],
    "orders": ["id", "user_id", "product_name", "amount", "order_date", "status"],
    "products": ["id", "name", "price", "category", "stock"],
    "transactions": ["id", "user_id", "amount", "date", "type"]
}

In [7]:
DANGEROUS_KEYWORDS = [
    "DROP", "DELETE", "ALTER", "TRUNCATE",
    "INSERT", "UPDATE", "EXEC", "EXECUTE"
]

DANGEROUS_PATTERNS = [
    ";",
    "--",
    "/*",
    "*/",
    "xp_",
    "sp_"
]

def is_safe(text):
    text_upper = text.upper()

    for keyword in DANGEROUS_KEYWORDS:
        if keyword in text_upper:
            return False

    for pattern in DANGEROUS_PATTERNS:
        if pattern in text:
            return False

    return True

In [8]:
def get_table_name(text):

    text = text.lower()

    if any(word in text for word in ["user","users","customer","account"]):
        return "users"

    if any(word in text for word in ["order","orders","purchase"]):
        return "orders"

    if any(word in text for word in ["product","products","item"]):
        return "products"

    if any(word in text for word in ["transaction","payment"]):
        return "transactions"

    return "users"

In [9]:
def get_column_names(text):

    columns = []

    mapping = {
        "name":["name","username"],
        "email":["email","mail"],
        "signup_date":["signup","joined","registered"],
        "age":["age","years"],
        "amount":["amount","price","cost"],
        "status":["status","state"],
        "country":["country","location"]
    }

    for column,keywords in mapping.items():
        if any(k in text.lower() for k in keywords):
            columns.append(column)

    if not columns:
        columns.append("*")

    return columns

In [10]:
def extract_conditions(text):

    conditions = []

    amount_match = re.search(r'greater than (\d+)',text)

    if amount_match:
        conditions.append(f"amount > {amount_match.group(1)}")

    if "last week" in text:
        conditions.append("DATE(date) >= DATE('now','-7 days')")

    if "last month" in text:
        conditions.append("DATE(signup_date) >= DATE('now','-1 month')")

    return conditions

In [11]:
def generate_sql(text):

    if not is_safe(text):
        return "Security Error: Unsafe query detected"

    table = get_table_name(text)

    columns = ",".join(get_column_names(text))

    conditions = extract_conditions(text)

    query = f"SELECT {columns} FROM {table}"

    if conditions:
        query += " WHERE " + " AND ".join(conditions)

    query += " LIMIT 10"

    return query

In [12]:
query_history = []

In [13]:
def export_pdf():

    filename = "queries.pdf"

    c = canvas.Canvas(filename)

    y = 800

    for i,q in enumerate(query_history):
        c.drawString(50,y,f"{i+1}. {q}")
        y -= 30

    c.save()

    return filename

In [14]:
def whatsapp_share():

    text = "\n".join(query_history[-5:])

    encoded = urllib.parse.quote(text)

    link = f"https://wa.me/?text={encoded}"

    return link

In [15]:
def chat(user_input):

    sql = generate_sql(user_input)

    query_history.append(sql)

    return sql

In [16]:
with gr.Blocks() as demo:

    gr.Markdown("# SQL Query Generator")

    user_input = gr.Textbox(label="Enter your query")

    output = gr.Textbox(label="Generated SQL")

    btn = gr.Button("Generate SQL")

    btn.click(chat,inputs=user_input,outputs=output)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c5b341ba24d2b60f5f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
